In [11]:
from game import Game
from collections import deque
import heapq
import time
import os
import re

## Load maps

In [12]:
def extract_map_files(directory):
    pattern = re.compile(r'^map(\d+)\.txt$')
    map_file_indices = []

    for file_name in os.listdir(directory):
        match = pattern.match(file_name)
        if match:
            map_file_indices.append(match.group(1))

    return [int(idx) for idx in map_file_indices]

def is_valid_input(map, indices, algorithm, solvers):
    valid_input = True
    if map not in indices:
        print(f"Map index out of range. Please choose within range {min(indices)} to {max(indices)}")
        valid_input = False
    if algorithm not in solvers.keys():    
        print(f"{algorithm} is not a defined algorithm. Please choose from", *[f"{solver} ({i+1})  " for i, solver in enumerate(solvers.keys())])
        valid_input = False
    return valid_input

def load_map(map_index):  
    file_name = "map" + str(map_index) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    return game_map

map_file_indices = extract_map_files("./assets/maps/")

## Tutorial

In [13]:
print("This is an example of the game map:")
map = load_map(2)
game = Game(map)
game.display_map()

This is an example of the game map:
W	P1	H	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [14]:
game.get_box_locations()

[(2, 3), (3, 2), (4, 3)]

In [15]:
game.get_goal_locations()

[(1, 3), (3, 1), (5, 3)]

In [16]:
game.get_player_position()

(0, 2)

- W : Wall
- H : Human
- B : Box
- P : Portal
- G : Goal

In [17]:
for direction in ['U', 'D', 'R', 'L']:
    result = game.apply_move(direction)
    print(f"Move {direction} is valid: {result}")
    if result:
        game.display_map()

Move U is valid: False
Move D is valid: False
Move R is valid: False
Move L is valid: True
W	P1	.	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	H	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [18]:
game.apply_move('U')
game.display_map()

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	H	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [19]:
game.apply_moves(['D', 'L', 'R', 'D']) 
game.display_map()
print("Is game won?", game.is_game_won())

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	.	W	W	W
W	G2/B2	.	.	P1	W	W
W	W	W	H	W	W	W
W	W	W	G3/B3	W	W	W
W	W	W	W	W	W	W
Is game won? True


## Solvers

### BFS

In [27]:
# TODO: Must return moves (if there is no solution return None), number of visited states
dir_map = {'U': (-1, 0), 'D': (1, 0), 'L': (0, -1), 'R': (0, 1)}
def solver_bfs(game_map):
    game = Game(game_map)
    queue = deque([(game.get_player_position(), tuple(game.get_box_locations()), "")])
    mark = set()
    while queue:
        mike, boxes, path = queue.popleft()
        if boxes == tuple(game.get_goal_locations()):
            return path, len(path)
        s = (mike, boxes)
        if s in mark:
            continue
        mark.add(s)
        for d in dir_map:
            game.set_player_position(mike)
            game.set_box_positions(list(boxes))
            if game.apply_move(d):
                neeew = (game.get_player_position(), tuple(game.get_box_locations()), path + d)
                queue.append(neeew)
    return "No solution", 0

### DFS

In [28]:
# TODO: Must return moves, number of visited states
def solver_dfs(game_map):
    game = Game(game_map)
    stack = [(game.get_player_position(), tuple(game.get_box_locations()), "")]
    mark = set()
    while stack:
        mike, boxes, path = stack.pop()
        if boxes == tuple(game.get_goal_locations()):
            return path, len(path)
        s = (mike, boxes)
        if s in mark:
            continue
        mark.add(s)
        for d in dir_map:
            game.set_player_position(mike)
            game.set_box_positions(list(boxes))
            if game.apply_move(d):
                neew = (game.get_player_position(), tuple(game.get_box_locations()), path + d)
                stack.append(neew)
    return "No solution", 0


### IDS

In [69]:
# TODO: Must return moves, number of visited states
def solver_ids(game_map):
    depth = 0
    while 1:
        mark = set()
        path = f_rec(Game(game_map), depth, mark, "")
        if path is not None:
            return path, len(path)
        depth += 1

def f_rec(game, depth, mark, path):
    mike = game.get_player_position()
    boxes = tuple(game.get_box_locations())
    if boxes == tuple(game.get_goal_locations()):
        return path
    if depth == 0:
        return None
    s = (mike, boxes)
    if s in mark:
        return None
    mark.add(s)
    for d in dir_map:
        game.set_player_position(mike)
        game.set_box_positions(list(boxes))
        if game.apply_move(d):
            neew = f_rec(game, depth - 1, mark, path + d)
            if neew is not None:
                return neew
    mark.remove(s)
    return None, 0

### A*

In [38]:
def MNH(first_point, second_point):
    x1 = first_point[0]
    x2 = first_point[1]
    y1 = second_point[0]
    y2 = second_point[1]
    manhatan = abs(x1 - y1) + abs(x2 - y2)
    return manhatan

def heuristic_1(game_map):
    game = Game(game_map)
    goals = game.get_goal_locations()
    def temp_function(now):
        ans = 0
        for i in range(len(now[1])):
            ans += MNH(now[0], now[1][i]) + MNH(now[1][i], goals[i])
        return ans
    return temp_function
def PO(p1, p2, portals):
    H0 = MNH(p1, p2)
    for portal in portals:
        H1 = min(MNH(p1, portal[0]) + MNH(portal[1], p2),MNH(p1, portal[1]) + MNH(portal[0], p2))
        H0 = min(H0, H1)
    return H0

def heuristic_2(game):
    goals = game.get_goal_locations()
    portals = game.get_portal_locations()
    def temp_function(s):
        mike, boxes = s
        if not boxes:
            return 0
        dis = []
        for box in boxes:
            distance = PO(mike, box, portals)
            dis.append(distance)
        mn1 = min(dis)
        mn2 = 0
        for i in range(len(boxes)):
            mn2 += PO(boxes[i], goals[i], portals)
        ans = mn1 + mn2
        return ans
    return temp_function
def solver_astar(game_map, heuristic_func=heuristic_2, weight=1):
    game = Game(game_map)
    temp_function = heuristic_func(game)
    H0 = (game.get_player_position(), tuple(game.get_box_locations()))
    h = temp_function(H0)
    goals = game.get_goal_locations()
    g = 0
    f = g + weight * h
    nodes = [(f, g, H0, "")]
    mark = {H0: g}
    while nodes:
        node = heapq.heappop(nodes)
        f = node[0]
        g = node[1]
        s = node[2]
        path = node[3]
        mike, boxes = s
        if list(boxes) == goals:
            return path, len(path)
        neighbors = []
        for d in dir_map:
            game_copy = Game(game_map)
            game_copy.set_player_position(mike)
            game_copy.set_box_positions(list(boxes))
            if game_copy.apply_move(d):
                neew = (game_copy.get_player_position(), tuple(game_copy.get_box_locations()))
                neighbors.append((d, neew))
        for d, neew in neighbors:
            now = g + 1
            if neew not in mark or now < mark[neew]:
                mark[neew] = now
                ff = now + weight * temp_function(neew)
                heapq.heappush(nodes, (ff, now, neew, path + d))
                
    return None, 0

## Solve

In [64]:
SOLVERS = {
    "BFS": solver_bfs,
    "DFS": solver_dfs,
    "IDS": solver_ids,
    "A*": solver_astar
}

In [65]:
def solve(map, method):  
    
    if not is_valid_input(map, map_file_indices, method, SOLVERS):
        return
    
    file_name = "map" + str(map) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    
    start_time = time.time()
    moves, numof_visited_states = SOLVERS[method](game_map)
    end_time = time.time()
    print(f"{method} took {round(end_time - start_time, 2)} seconds on map {map} and visited {numof_visited_states} states.")
    
    if moves is None:
        print("No Solution Found!")
    else:
        print(f"{len(moves)} moves were used: {moves}")
            

In [71]:
solve(8, "IDS") # Solve map 1 using BFS

IDS took 0.0 seconds on map 8 and visited 2 states.
2 moves were used: (None, 0)


In [42]:
def solve_all():
    for map in range(min(map_file_indices), max(map_file_indices) + 1):
        for method in SOLVERS.keys():
            solve(map, method)
            

In [70]:
solve_all() # Solve all maps using all methods

BFS took 0.0 seconds on map 1 and visited 7 states.
7 moves were used: UDDULRR
DFS took 0.0 seconds on map 1 and visited 7 states.
7 moves were used: RLLRDUU
IDS took 0.0 seconds on map 1 and visited 2 states.
2 moves were used: (None, 0)
A* took 0.0 seconds on map 1 and visited 7 states.
7 moves were used: DURLLRU
BFS took 0.0 seconds on map 2 and visited 6 states.
6 moves were used: LUDDUL
DFS took 0.0 seconds on map 2 and visited 6 states.
6 moves were used: LLRDUU
IDS took 0.0 seconds on map 2 and visited 2 states.
2 moves were used: (None, 0)
A* took 0.0 seconds on map 2 and visited 6 states.
6 moves were used: LDULRU
BFS took 0.0 seconds on map 3 and visited 13 states.
13 moves were used: ULDDUUUURDDDD
DFS took 0.0 seconds on map 3 and visited 13 states.
13 moves were used: ULDDUUUURDDDD
IDS took 0.0 seconds on map 3 and visited 2 states.
2 moves were used: (None, 0)
A* took 0.01 seconds on map 3 and visited 13 states.
13 moves were used: ULDDUUUURDDDD
BFS took 0.0 seconds on map

KeyboardInterrupt: 